# Test classifiers and results on the shipping data: SILVA database

In [1]:
import pandas as pd
import os

import tkinter as tk

import qiime2 as q2

%matplotlib inline

First, I'll retrain a classifier

In [2]:
import os
import qiime2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the working directory
#wd = '/Users/meyeanni/Desktop/git_sourdough/SourdoughFlow/LP5/Citizen_reports/LP5_LP7_integration/SILVA/new_12.25'
wd = '/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25'
# Change to the working directory
os.chdir(wd)

# Verify current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25


In [3]:
from qiime2 import (Artifact,
                    Metadata as qmd)

from qiime2.plugins import (cutadapt,
                            demux,
                            feature_table as qft,
                            taxa as q2t,)

from qiime2 import Metadata
from qiime2 import Visualization

from qiime2.plugins.feature_table.methods import (merge_seqs, merge, filter_seqs, filter_samples, filter_features) 
import qiime2.plugins.feature_classifier.actions as feature_classifier_actions
import qiime2.plugins.metadata.actions as metadata_actions
import qiime2.plugins.taxa.actions as taxa_actions
import qiime2.plugins.phylogeny.actions as phylogeny_actions
from qiime2.plugins.fragment_insertion.methods import sepp

/home/meyeanni/miniconda3/envs/qiime2-amplicon-2025.7/lib/python3.10/site-packages/q2_demux/_summarize/_visualizer.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
pwd

'/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25'

get new, clean silva data:

In [12]:
! qiime rescript get-silva-data \
    --p-version '138.2' \
    --p-target 'SSURef_NR99' \
    --p-include-species-labels \
    --o-silva-sequences silva-138.2-ssu-nr99-seqs.qza \
    --o-silva-taxonomy silva-138.2-ssu-nr99-tax.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[RNASequence] to: silva-138.2-ssu-nr99-seqs.qza
Saved FeatureData[Taxonomy] to: silva-138.2-ssu-nr99-tax.qza


RNA->DNA conversion

In [13]:
! qiime rescript reverse-transcribe \
    --i-rna-sequences silva-138.2-ssu-nr99-seqs.qza \
    --o-dna-sequences silva-138.2-ssu-nr99-seqs-dna.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna.qza


clean seqeunces, remove bad quality sequences

In [14]:
! qiime rescript cull-seqs \
 --i-sequences silva-138.2-ssu-nr99-seqs-dna.qza \
 --o-clean-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna-cleaned.qza


filter by length

In [15]:
!qiime rescript filter-seqs-length-by-taxon \
    --i-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned.qza \
    --i-taxonomy silva-138.2-ssu-nr99-tax.qza \
    --p-labels Archaea Bacteria Eukaryota \
    --p-min-lens 900 1200 1400 \
    --o-filtered-seqs silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered.qza \
    --o-discarded-seqs silva-138.2-ssu-nr99-seqs-dna-cleaned-discarded.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered.qza
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna-cleaned-discarded.qza


filter taxonomy based on seq ids

In [16]:
! qiime rescript filter-taxa \
    --i-taxonomy silva-138.2-ssu-nr99-tax.qza \
    --m-ids-to-keep-file silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered.qza \
    --o-filtered-taxonomy silva-138.2-ssu-nr99-tax-filtered.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Taxonomy] to: silva-138.2-ssu-nr99-tax-filtered.qza


extract primer regions

In [17]:
! qiime feature-classifier extract-reads \
  --i-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered.qza \
  --p-f-primer GTGYCAGCMGCCGCGGTAA \
  --p-r-primer GGACTACNVGGGTWTCTAAT \
  --p-n-jobs 2 \
  --p-read-orientation 'forward' \
  --o-reads silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-515f-806r.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-515f-806r.qza


dereplicate seqeunces and taxonomy

In [18]:
! qiime rescript dereplicate \
  --i-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-515f-806r.qza \
  --i-taxa silva-138.2-ssu-nr99-tax-filtered.qza \
  --p-mode 'uniq' \
  --o-dereplicated-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-derep-515f-806r.qza \
  --o-dereplicated-taxa silva-138.2-ssu-nr99-tax-filtered-derep-515f-806r.qza

fatal: bad revision 'HEAD'
fatal: bad revision 'HEAD'
Saved FeatureData[Sequence] to: silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-derep-515f-806r.qza
Saved FeatureData[Taxonomy] to: silva-138.2-ssu-nr99-tax-filtered-derep-515f-806r.qza


fit classifier

In [7]:
! qiime rescript evaluate-fit-classifier \
    --i-sequences silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-derep-515f-806r.qza \
    --i-taxonomy silva-138.2-ssu-nr99-tax-filtered-derep-515f-806r.qza \
    --o-classifier silva-138.2-ssu-nr99-classifier-515f-806r.qza \
    --o-evaluation silva-138.2-ssu-nr99-classifier-evaluation.qzv \
    --o-observed-taxonomy silva-138.2-ssu-nr99-predicted-taxonomy.qza

Saved TaxonomicClassifier to: silva-138.2-ssu-nr99-classifier-515f-806r.qza
Saved Visualization to: silva-138.2-ssu-nr99-classifier-evaluation.qzv
Saved FeatureData[Taxonomy] to: silva-138.2-ssu-nr99-predicted-taxonomy.qza


-> it took 203 minutes

## now classify the test data (shipping data) with the silva classifier

In [4]:
fn = '/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25/silva-138.2-ssu-nr99-classifier-515f-806r.qza'
classifier = Artifact.load(fn)

In [7]:
pwd

'/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25'

In [5]:
# this is the `FeatureData[Sequence]` 

rep_seqs = q2.Artifact.load('/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/test_data/bacteria-seqs-filt-shipping-ASVs-380.qza')

In [6]:
pwd

'/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/new_12.25'

In [7]:
wd = '/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests'
# Change to the working directory
os.chdir(wd)

# Verify current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests


In [24]:
taxonomy, = feature_classifier_actions.classify_sklearn(
    classifier=classifier,
    reads=rep_seqs,
    confidence=0,
    n_jobs=1  # Make sure this is a valid number of threads for your environment
)

taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)

taxonomy.save('test_data/classifications/taxonomy_silva_conf0.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_conf0.qzv')

'test_data/classifications/taxonomy_silva_conf0.qzv'

In [25]:
taxonomy, = feature_classifier_actions.classify_sklearn(
    classifier=classifier,
    reads=rep_seqs,
    confidence=0.7,
    n_jobs=1  # Make sure this is a valid number of threads for your environment
)

taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)

taxonomy.save('test_data/classifications/taxonomy_silva_conf0.7.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_conf0.7.qzv')

'test_data/classifications/taxonomy_silva_conf0.7.qzv'

also, use an average weighted, pre-trained classifier : https://www.arb-silva.de/archive/release_138.2/QIIME2/2025.7/SSU/V4-515f-806r/weighted/average

In [8]:
fn = '/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests/SILVA/weighted_average/SILVA138.2_SSURef_NR99_weighted_classifier_V4-515f-806r_average.qza'
classifier = Artifact.load(fn)

reclassify with this average weighted classifier:

In [9]:
taxonomy, = feature_classifier_actions.classify_sklearn(
    classifier=classifier,
    reads=rep_seqs,
    confidence=0,
    n_jobs=1  # Make sure this is a valid number of threads for your environment
)

taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)

taxonomy.save('test_data/classifications/taxonomy_silva_average_weight_conf0.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_average_weight_conf0.qzv')

'test_data/classifications/taxonomy_silva_average_weight_conf0.qzv'

In [10]:
taxonomy, = feature_classifier_actions.classify_sklearn(
    classifier=classifier,
    reads=rep_seqs,
    confidence=0.7,
    n_jobs=1  # Make sure this is a valid number of threads for your environment
)

taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)

taxonomy.save('test_data/classifications/taxonomy_silva_average_weight_conf0.7.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_average_weight_conf0.7.qzv')

'test_data/classifications/taxonomy_silva_average_weight_conf0.7.qzv'

In [11]:
pwd

'/home/meyeanni/cloud/meyeanni/LP4/artifacts/16S/classifier_tests'

In [12]:
#import the reference taxonomy and reference reads:
ref_taxonomy = q2.Artifact.load('SILVA/new_12.25/silva-138.2-ssu-nr99-tax-filtered-derep-515f-806r.qza')
ref_seqs = q2.Artifact.load('SILVA/new_12.25/silva-138.2-ssu-nr99-seqs-dna-cleaned-filtered-derep-515f-806r.qza')

In [13]:
#now with classify-consensus-blast
taxonomy,hits = feature_classifier_actions.classify_consensus_blast(
    reference_reads=ref_seqs,
    reference_taxonomy=ref_taxonomy,
    query=rep_seqs,
    perc_identity=0.99,
    maxaccepts=1,
    min_consensus=0.51,
    evalue=1e-100,
    query_cov=0.99
)
taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)
taxonomy.save('test_data/classifications/taxonomy_silva_blast99_1.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_blast99_1.qzv')

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: blastn -query /scratch/meyeanni/tmp/qiime2/meyeanni/data/02cb3db1-7834-4a42-b73a-a65bd02326ba/data/dna-sequences.fasta -evalue 1e-100 -strand both -outfmt 6 -perc_identity 99.0 -qcov_hsp_perc 99.0 -num_threads 1 -max_target_seqs 1 -out /scratch/meyeanni/tmp/qiime2/meyeanni/processes/1072367-1765965243.74@meyeanni/tmp/q2-OutPath-chx66b0r -subject /scratch/meyeanni/tmp/qiime2/meyeanni/data/36825885-416c-4a00-a24c-12716c38cc49/data/dna-sequences.fasta



'test_data/classifications/taxonomy_silva_blast99_1.qzv'

In [14]:
#now with classify-consensus-vsearch
taxonomy,hits = feature_classifier_actions.classify_consensus_vsearch(
    reference_reads=ref_seqs,
    reference_taxonomy=ref_taxonomy,
    query=rep_seqs,
    perc_identity=0.99,
    maxaccepts=1,
    min_consensus=0.51,
    query_cov=0.99,
    
)
taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)
taxonomy.save('test_data/classifications/taxonomy_silva_vsearch100.qza')
taxonomy_viz.save('test_data/classifications/taxonomy_silva_vsearch100.qzv')

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: vsearch --usearch_global /scratch/meyeanni/tmp/qiime2/meyeanni/data/02cb3db1-7834-4a42-b73a-a65bd02326ba/data/dna-sequences.fasta --id 0.99 --query_cov 0.99 --strand both --maxaccepts 1 --maxrejects 0 --db /scratch/meyeanni/tmp/qiime2/meyeanni/data/36825885-416c-4a00-a24c-12716c38cc49/data/dna-sequences.fasta --threads 1 --output_no_hits --blast6out /scratch/meyeanni/tmp/qiime2/meyeanni/processes/1072367-1765965243.74@meyeanni/tmp/q2-OutPath-jwanyztc



vsearch v2.22.1_linux_x86_64, 91.6GB RAM, 12 cores
https://github.com/torognes/vsearch

Reading file /scratch/meyeanni/tmp/qiime2/meyeanni/data/36825885-416c-4a00-a24c-12716c38cc49/data/dna-sequences.fasta 100%
89354958 nt in 321134 seqs, min 54, max 1784, avg 278
Masking 100%
Counting k-mers 100%
Creating k-mer index 100%
Searching 100%
Matching unique query sequences: 8 of 8 (100.00%)


'test_data/classifications/taxonomy_silva_vsearch100.qzv'